# Metax (MACA) environment check

Run this inside a Metax DevEnvironment to confirm what the image carries and that the
GPU can actually be used by the account the environment runs as.

The device count and the device names report the vendor's configured visible-device set.
They can both be correct while every open of the device fails, so they are printed rather
than asserted. The last cell is the one that proves access.

Give it memory. The first call that really touches a device initializes the MACA driver in
this process, which costs several GiB of host memory, so an environment of much under
16 GiB is OOM-killed part-way through — typically at the last cell, with everything above
it already printed. That is the environment's memory limit rather than a broken GPU.


In [ ]:
# The interpreter this kernel runs on. In the MACA images it is the vendor's
# /opt/conda python, which is the one that carries torch.
from platform import python_version
print(f"Python Version: {python_version()}")


In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
# The CUDA-compatibility level this MACA build is a port of.
print(f"CUDA-Compatible Version: {torch.version.cuda}")


In [ ]:
# The vendor's own stack version.
import pymxsml
pymxsml.mxSmlInit()
print(f"MetaX MACA Version: {pymxsml.mxSmlGetMacaVersion()}")


In [ ]:
# What the environment was configured to see.
count = torch.cuda.device_count()
print(f"Total Physical GPUs: {count}")
print([torch.cuda.get_device_name(i) for i in range(count)])


In [ ]:
# The access check. Both of these need the device to be openable by this account,
# which is what the cell above cannot show: a non-root environment can report the
# right count and then fail here. If it does, the node's container runtime is not
# handing this account the device.
print(f"CUDA Available: {torch.cuda.is_available()}")

x = torch.randn(256, 256, device="cuda")
print(f"256x256 matmul ok, sum = {float((x @ x).sum())}")
